## Statistical analysis of potential improvements at different training sizes for substituition datasets

In [1]:
import pandas as pd
from model_comparison import rm_tukey_hsd

### Read data

In [2]:
df_input = pd.read_csv("../simulation_outputs/toxinpred3/simulation_summary_all.csv")

# Make name shorter
df_input.loc[df_input["Model"] == "GaussianProcessClassifier", "Model"] = "GPC"
df_input.loc[df_input["Model"] == "ExtraTrees", "Model"] = "ET"

# Add simplified foundation column for grouping
df_input["Foundation Simple"] = df_input.apply(
    lambda row: ("" if not row['Baseline'] else " + Descriptors"),
    axis=1
)

df_input["Model Label"] = (
    df_input.apply(
        lambda row: f"{row['Model']}" + row['Foundation Simple'],
        axis=1
    )
)

df_input = df_input[
    df_input['Model Label'].isin([
        "ET",
        "ET + Descriptors"
    ])
]

df_input["method"] = df_input['Model Label']
df_input["cv_cycle"] = df_input["Seed"]
df_input["split"] = ["Random"]*len(df_input["method"])

datasets = ["toxinpred3"]

train_sizes = df_input["Train Size"].unique()

df_all = {data: {} for data in datasets}
for data in datasets:
    for size in train_sizes:
        df_all[data][size] = df_input[(df_input['Train Size'] == size) & (df_input["Dataset"] == data)]

# Defining column names for metrics
metric_ls = ["Test Accuracy", 
            "Test Balanced Accuracy", 
            "Test F1 (weighted)",
            "Test MCC",
            "Test ROC AUC"]

### Evaluation summary across datasets per training size

Takes the mean p-value and mean size of difference across datasets per training size

In [3]:
df_results = pd.DataFrame(columns=["Train Size", "Metric", "p-adj mean value", "p-adj std value", "effect size mean value", "effect size std value"])
df_results_all = pd.DataFrame(columns=["Train Size", "Dataset", "Metric", "p-adj", "effect size"])
for metric in metric_ls:
    for size in train_sizes:
        p_values = []
        effect_size = []
        for data in datasets:
            print(f"Dataset :{data} | Training size: {size}")
            results = rm_tukey_hsd(df_all[data][size], metric, "Model Label")[0]
            p_values.append(results["p-adj"].iloc[0])
            effect_size.append(results["meandiff"].iloc[0])
            df_results_all.loc[len(df_results_all)] = [size, data, metric, results["p-adj"].iloc[0], results["meandiff"].iloc[0]]

        df_results.loc[len(df_results)] = [size, metric, pd.Series(p_values).mean(), pd.Series(p_values).std(), pd.Series(effect_size).mean(), pd.Series(effect_size).std()]

df_results = df_results.sort_values(by="Train Size").reset_index(drop=True)
df_results.to_csv("results/toxinpred3_results_summary.csv", index=False)
df_results_all.to_csv("results/toxinpred3_results.csv", index=False)

Dataset :toxinpred3 | Training size: 500
Dataset :toxinpred3 | Training size: 75
Dataset :toxinpred3 | Training size: 200
Dataset :toxinpred3 | Training size: 100
Dataset :toxinpred3 | Training size: 50
Dataset :toxinpred3 | Training size: 350
Dataset :toxinpred3 | Training size: 20
Dataset :toxinpred3 | Training size: 500
Dataset :toxinpred3 | Training size: 75
Dataset :toxinpred3 | Training size: 200
Dataset :toxinpred3 | Training size: 100
Dataset :toxinpred3 | Training size: 50
Dataset :toxinpred3 | Training size: 350
Dataset :toxinpred3 | Training size: 20
Dataset :toxinpred3 | Training size: 500
Dataset :toxinpred3 | Training size: 75
Dataset :toxinpred3 | Training size: 200
Dataset :toxinpred3 | Training size: 100
Dataset :toxinpred3 | Training size: 50
Dataset :toxinpred3 | Training size: 350
Dataset :toxinpred3 | Training size: 20
Dataset :toxinpred3 | Training size: 500
Dataset :toxinpred3 | Training size: 75
Dataset :toxinpred3 | Training size: 200
Dataset :toxinpred3 | Trai